## Import

In [1]:
import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30   # same Optuna budget for every model
INNER_K = 3     # inner folds used for tuning
os.makedirs("weights_V6_0", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


## Dataloader

In [4]:
def prep_split(tr_idx, te_idx):
    """Standardise using the training rows only (no missing values left in the regression set)."""
    sc = StandardScaler().fit(X_full[tr_idx])
    return sc.transform(X_full[tr_idx]), sc.transform(X_full[te_idx]), y_raw[tr_idx], y_raw[te_idx]

# same 80/20 split as every earlier version (seed 42)
idx_all = np.arange(len(y_raw))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
Xtr, Xte, ytr, yte = prep_split(tr_idx, te_idx)
print(f"train {Xtr.shape} test {Xte.shape}")

train (1769, 13) test (443, 13)


## Model

In [5]:
from xgboost import XGBRegressor
SPECS = {
 "XGBoost": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 100, 600, step=100), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         d=t.suggest_int("d", 2, 8), sub=t.suggest_float("sub", 0.6, 1.0),
                         col=t.suggest_float("col", 0.5, 1.0), mcw=t.suggest_int("mcw", 1, 10),
                         lam=t.suggest_float("lam", 1e-2, 10, log=True)),
    make=lambda p, s: XGBRegressor(n_estimators=p["n"], learning_rate=p["lr"], max_depth=p["d"], subsample=p["sub"],
                         colsample_bytree=p["col"], min_child_weight=p["mcw"], reg_lambda=p["lam"],
                         n_jobs=1, random_state=s, verbosity=0)),
}

## Tuning

In [6]:
def inner_oof(spec, params, X, y, seed):
    """Out-of-fold predictions on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in KFold(INNER_K, shuffle=True, random_state=seed).split(X):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = est.predict(X[b])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) minimising RMSE of the inner out-of-fold predictions."""
    if spec.get("fixed"):
        return {}
    def objective(trial):
        return float(np.sqrt(mean_squared_error(y, inner_oof(spec, spec["space"](trial), X, y, seed))))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    pred = est.predict(Xb)
    mse = mean_squared_error(yb, pred)
    return {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse}, pred

In [7]:
best_params = {}
for name, spec in SPECS.items():
    t0 = time.time()
    best_params[name] = tune(spec, Xtr, ytr, SEED)
    print(f"{name:32s} tuned in {time.time() - t0:6.1f}s | {best_params[name]}")
json.dump(best_params, open("weights_V6_0/regression_xgboost_best_params.json", "w"), indent=2, default=str)

XGBoost                          tuned in   20.2s | {'n': 400, 'lr': 0.08800772495767324, 'd': 6, 'sub': 0.9116805319122259, 'col': 0.8979888490578186, 'mcw': 3, 'lam': 0.7192128970520779}


## Training

In [8]:
results, scores = [], {}
for name, spec in SPECS.items():
    row, s = fit_eval(spec, best_params[name], Xtr, ytr, Xte, yte, SEED)
    scores[name] = s
    results.append(dict(model=name, tier=spec["tier"], selection="inner validation",
                        **{k: round(float(v), 4) for k, v in row.items()}, params=best_params[name]))
    print(f"{name:32s} R2 = {row['r2']:.4f}")

XGBoost                          R2 = 0.8306


## Evaluation

In [9]:
table = pd.DataFrame(results).sort_values("r2", ascending=False)
print(table.drop(columns=["params", "selection"]).to_string(index=False))
json.dump({"n_test": int(len(yte)), "seed": SEED, "n_trials": N_TRIALS, "inner_k": INNER_K, "models": results},
          open("weights_V6_0/regression_xgboost_test_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V6_0/regression_xgboost_test_scores.npz", y_true=yte, **{k: v for k, v in scores.items()})

  model              tier     r2      rmse       mae          mse
XGBoost curriculum family 0.8306 8969.4745 5114.3778 8.045147e+07
